In [3]:
from pathlib import Path
import numpy as np, pandas as pd
from tensorflow import keras

# project root = folder above "notebooks"
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Project root:", ROOT)

df = pd.read_parquet(ROOT / "data" / "m100_reduced.parquet")
print("Data shape:", df.shape)

model = keras.models.load_model(ROOT / "models" / "lstm_ae.keras")
print("Model input:", model.input_shape, "| output:", model.output_shape)

Project root: d:\!!DATA HP LAPTOP!!\d\5 SEM\DL PROJECT
Data shape: (751363, 25)
Model input: (None, 24, 22) | output: (None, 24, 22)


In [4]:
from pathlib import Path
import numpy as np, pandas as pd
from tensorflow import keras

# project root = folder above "notebooks"
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Project root:", ROOT)

df = pd.read_parquet(ROOT / "data" / "m100_reduced.parquet")
print("Data shape:", df.shape)

model = keras.models.load_model(ROOT / "models" / "lstm_ae.keras")
print("Model input:", model.input_shape, "| output:", model.output_shape)

Project root: d:\!!DATA HP LAPTOP!!\d\5 SEM\DL PROJECT
Data shape: (751363, 25)
Model input: (None, 24, 22) | output: (None, 24, 22)


In [5]:
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import StandardScaler

# reload fresh each time, so running this cell twice is safe
df = pd.read_parquet(ROOT / "data" / "m100_reduced.parquet")

W = 24
feat = [c for c in df.columns if c not in ('timestamp', 'node', 'label', 'sensor_na', 'lab', 'split')]
df = df.sort_values(['node', 'timestamp']).reset_index(drop=True)
df['lab'] = df['label'].fillna(-1).astype('int8')
df[feat] = df.groupby('node')[feat].ffill(limit=4)

df['split'] = ''
for n, g in df.groupby('node'):
    k1, k2 = int(0.7 * len(g)), int(0.8 * len(g))
    df.loc[g.index[:k1], 'split'] = 'train'
    df.loc[g.index[k1:k2], 'split'] = 'val'
    df.loc[g.index[k2:], 'split'] = 'test'

fit_rows = df[(df.split == 'train') & (df.lab == 0)].dropna(subset=feat)
scaler = StandardScaler().fit(fit_rows[feat])
df[feat] = np.clip(scaler.transform(df[feat]), -5, 5).astype('float32')

def make_windows(split, stride):
    X, y, nodes_ = [], [], []
    for n, g in df[df.split == split].groupby('node'):
        if len(g) < W: continue
        F = g[feat].to_numpy(dtype='float32'); L = g['lab'].to_numpy()
        T = ((g['timestamp'] - g['timestamp'].iloc[0]).dt.total_seconds() / 60).to_numpy()
        Fw = sliding_window_view(F, (W, F.shape[1]))[:, 0]
        Lw = sliding_window_view(L, W); Tw = sliding_window_view(T, W)
        keep = (~np.isnan(Fw).any(axis=(1, 2))
                & ((Tw[:, -1] - Tw[:, 0]) == (W - 1) * 15)
                & np.isin(Lw, [0, 2]).all(axis=1))
        sel = np.zeros_like(keep); sel[::stride] = True; keep &= sel
        X.append(Fw[keep]); y.append((Lw[keep] == 2).any(axis=1)); nodes_.append(np.full(keep.sum(), n))
    return np.concatenate(X), np.concatenate(y), np.concatenate(nodes_)

Xtr, ytr, ntr = make_windows('train', 4); Xtr = Xtr[~ytr]
Xva, yva, nva = make_windows('val', 4);   Xva = Xva[~yva]
Xte, yte, nte = make_windows('test', 2)

print(Xtr.shape, Xva.shape, Xte.shape, int(yte.sum()))
print(len(feat), feat)
print(pd.Series(nte[yte]).value_counts().sort_index().to_dict())

(101924, 24, 22) (17926, 24, 22) (72252, 24, 22) 761
22 ['ambient', 'fan0_0', 'fan0_1', 'fan1_0', 'fan1_1', 'fan2_0', 'fan2_1', 'fan3_0', 'fan3_1', 'fan_disk_power', 'p0_vdd_temp', 'p1_vdd_temp', 'dimm_temp', 'p0_core_temp', 'p1_core_temp', 'gpu_core_temp', 'gpu_mem_temp', 'p0_power', 'p1_power', 'total_power', 'ps0_input_power', 'ps1_input_power']
{7: 55, 8: 34, 9: 40, 10: 428, 11: 41, 13: 34, 14: 41, 16: 54, 17: 34}


In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score

def window_error(X, bs=1024):
    R = model.predict(X, batch_size=bs, verbose=0)
    return np.mean((X - R) ** 2, axis=(1, 2))

err_va = window_error(Xva)
err_te = window_error(Xte)

print(np.percentile(err_va, [50, 90, 95, 99, 99.9]).round(4))
print(np.median(err_te[~yte]).round(4), np.median(err_te[yte]).round(4))
print(round(roc_auc_score(yte, err_te), 3), round(average_precision_score(yte, err_te), 4), round(yte.mean(), 4))

[0.0023 0.0048 0.0085 0.1321 0.9676]
0.0027 0.0503
0.787 0.0581 0.0105


In [5]:
power   = ['p0_power', 'p1_power', 'total_power', 'ps0_input_power', 'ps1_input_power']
other   = ['fan_disk_power']
cooling = [c for c in feat if c not in power + other]
print("group sizes (cooling, power, other):", len(cooling), len(power), len(other))

R = model.predict(Xte, batch_size=1024, verbose=0)
sens_err = ((Xte - R) ** 2).mean(axis=1)          # shape (windows, 22): error per sensor
print("check, matches earlier window error:", np.allclose(sens_err.mean(axis=1), err_te, rtol=1e-3))

idx = {c: i for i, c in enumerate(feat)}
def share(cols):
    return sens_err[:, [idx[c] for c in cols]].sum(axis=1) / sens_err.sum(axis=1)
sh = pd.DataFrame({'cooling': share(cooling), 'power': share(power), 'other': share(other)})

thr = np.percentile(err_va, 99)                    # the 99th-percentile threshold from validation
flag = err_te > thr
kind = np.select([~yte & ~flag, ~yte & flag, yte & flag, yte & ~flag],
                 ['normal, not flagged', 'normal, flagged (false alarm)', 'anomaly, flagged', 'anomaly, missed'])

out = sh.groupby(kind).mean().round(3)
out['windows'] = pd.Series(kind).value_counts()
print(out)
print("if error were spread evenly:  cooling", round(16/22, 3), " power", round(5/22, 3), " other", round(1/22, 3))

per_sensor = pd.DataFrame(sens_err, columns=feat)
for name in ['anomaly, flagged', 'normal, flagged (false alarm)']:
    m = per_sensor[kind == name]
    top = m.div(m.sum(axis=1), axis=0).mean().sort_values(ascending=False).head(5).round(3)
    print("\nTop 5 sensors, share of error:", name)
    print(top.to_string())

group sizes (cooling, power, other): 16 5 1
check, matches earlier window error: True


TypeError: Choicelist and default value do not have a common dtype: The DType <class 'numpy.dtypes._PyLongDType'> could not be promoted by <class 'numpy.dtypes.StrDType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes._PyLongDType'>)

In [ ]:
kind = np.select([~yte & ~flag, ~yte & flag, yte & flag, yte & ~flag],
                 ['normal, not flagged', 'normal, flagged (false alarm)', 'anomaly, flagged', 'anomaly, missed'], default='')
                 default='')

out = sh.groupby(kind).mean().round(3)
out['windows'] = pd.Series(kind).value_counts()
print(out)
print("if error were spread evenly:  cooling", round(16/22, 3), " power", round(5/22, 3), " other", round(1/22, 3))

per_sensor = pd.DataFrame(sens_err, columns=feat)
for name in ['anomaly, flagged', 'normal, flagged (false alarm)']:
    m = per_sensor[kind == name]
    top = m.div(m.sum(axis=1), axis=0).mean().sort_values(ascending=False).head(5).round(3)
    print("\nTop 5 sensors, share of error:", name)
    print(top.to_string())

                               cooling  power  other  windows
anomaly, flagged                 0.778  0.202  0.020      250
anomaly, missed                  0.630  0.365  0.005      511
normal, flagged (false alarm)    0.433  0.563  0.004     3752
normal, not flagged              0.680  0.311  0.009    67739
if error were spread evenly:  cooling 0.727  power 0.227  other 0.045

Top 5 sensors, share of error: anomaly, flagged
gpu_mem_temp       0.311
p1_core_temp       0.077
ps1_input_power    0.064
gpu_core_temp      0.055
fan1_0             0.048

Top 5 sensors, share of error: normal, flagged (false alarm)
ps1_input_power    0.137
total_power        0.131
ps0_input_power    0.123
gpu_mem_temp       0.122
gpu_core_temp      0.109


In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_fscore_support, confusion_matrix)

R_va = model.predict(Xva, batch_size=1024, verbose=0)
sens_va = ((Xva - R_va) ** 2).mean(axis=1)           # per-sensor error on validation windows

ci = [idx[c] for c in cooling]                        # the 16 cooling sensors
candidates = [('full (22 sensors)',          err_te,                     err_va),
              ('cooling only (16 sensors)',  sens_err[:, ci].mean(axis=1), sens_va[:, ci].mean(axis=1))]

rows = []
for name, s_te, s_va in candidates:
    t99 = np.percentile(s_va, 99)                     # threshold from validation only
    pred = s_te > t99
    p, r, f1, _ = precision_recall_fscore_support(yte, pred, average='binary', zero_division=0)
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    rows.append([name, round(roc_auc_score(yte, s_te), 3), round(average_precision_score(yte, s_te), 4),
                 round(p, 3), round(r, 3), fp, tp])
print(pd.DataFrame(rows, columns=['score', 'ROC-AUC', 'PR-AUC', 'precision', 'recall', 'false_pos', 'true_pos']).to_string(index=False))

                    score  ROC-AUC  PR-AUC  precision  recall  false_pos  true_pos
        full (22 sensors)    0.787  0.0581      0.062   0.329       3752       250
cooling only (16 sensors)    0.791  0.0794      0.077   0.329       3005       250


In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
                             precision_recall_fscore_support)

rows = []
for i, name in enumerate(feat):
    s_te, s_va = sens_err[:, i], sens_va[:, i]        # this one sensor's error on test / validation
    t99 = np.percentile(s_va, 99)                      # threshold from validation only
    pred = s_te > t99
    p, r, f1, _ = precision_recall_fscore_support(yte, pred, average='binary', zero_division=0)
    rows.append([name, round(roc_auc_score(yte, s_te), 3), round(average_precision_score(yte, s_te), 4),
                 round(p, 3), round(r, 3), round(accuracy_score(yte, pred), 3)])

rank = pd.DataFrame(rows, columns=['sensor', 'ROC-AUC', 'PR-AUC', 'precision', 'recall', 'accuracy'])
print(rank.sort_values('PR-AUC', ascending=False).to_string(index=False))
print("random PR-AUC would be about", round(yte.mean(), 4))

         sensor  ROC-AUC  PR-AUC  precision  recall  accuracy
         fan0_0    0.689  0.1120      0.179   0.311     0.978
         fan2_0    0.648  0.1117      0.034   0.464     0.856
         fan3_1    0.762  0.1101      0.024   0.428     0.810
         fan0_1    0.679  0.1073      0.299   0.294     0.985
         fan1_1    0.701  0.1067      0.069   0.386     0.938
        ambient    0.765  0.1029      0.098   0.332     0.961
 fan_disk_power    0.647  0.1025      0.080   0.389     0.946
         fan3_0    0.784  0.1020      0.060   0.535     0.906
         fan2_1    0.671  0.0973      0.024   0.430     0.813
   p1_core_temp    0.754  0.0848      0.093   0.340     0.958
    p0_vdd_temp    0.765  0.0765      0.080   0.330     0.953
   gpu_mem_temp    0.734  0.0729      0.055   0.297     0.939
    p1_vdd_temp    0.764  0.0722      0.095   0.311     0.962
         fan1_0    0.682  0.0695      0.081   0.382     0.948
      dimm_temp    0.728  0.0475      0.077   0.258     0.959
ps1_inpu

In [6]:
def runs(flag, node):
    out, i, n = [], 0, len(flag)
    while i < n:
        if flag[i]:
            j = i
            while j + 1 < n and flag[j + 1] and node[j + 1] == node[i]:
                j += 1
            out.append((i, j)); i = j + 1
        else:
            i += 1
    return out

def event_report(thr, score):
    pred = score > thr
    episodes = runs(yte, nte); alarms = runs(pred, nte)
    detected = int(sum(pred[s:e + 1].any() for s, e in episodes))
    true_bursts = int(sum(yte[s:e + 1].any() for s, e in alarms))
    return [len(episodes), detected, round(detected / len(episodes), 3),
            len(alarms), len(alarms) - true_bursts, round(true_bursts / max(len(alarms), 1), 3)]

candidates = [('full (22 sensors)', err_te, err_va),
              ('cooling only (16)', sens_err[:, ci].mean(axis=1), sens_va[:, ci].mean(axis=1))]
rows = []
for name, s_te, s_va in candidates:
    for rule, thr in [('95th', np.percentile(s_va, 95)),
                      ('99th', np.percentile(s_va, 99)),
                      ('mean+3std', s_va.mean() + 3 * s_va.std())]:
        rows.append([name, rule] + event_report(thr, s_te))

print(pd.DataFrame(rows, columns=['score', 'threshold', 'episodes', 'detected', 'event_recall',
                                  'alarm_bursts', 'false_bursts', 'burst_precision']).to_string(index=False))

NameError: name 'ci' is not defined

In [7]:
R_va = model.predict(Xva, batch_size=1024, verbose=0)
sens_va = ((Xva - R_va) ** 2).mean(axis=1)      # per-sensor error on validation windows
ci = [idx[c] for c in cooling]                    # positions of the 16 cooling sensors
print(sens_va.shape, len(ci))

(17926, 22) 16


In [8]:
def runs(flag, node):
    out, i, n = [], 0, len(flag)
    while i < n:
        if flag[i]:
            j = i
            while j + 1 < n and flag[j + 1] and node[j + 1] == node[i]:
                j += 1
            out.append((i, j)); i = j + 1
        else:
            i += 1
    return out

def event_report(thr, score):
    pred = score > thr
    episodes = runs(yte, nte); alarms = runs(pred, nte)
    detected = int(sum(pred[s:e + 1].any() for s, e in episodes))
    true_bursts = int(sum(yte[s:e + 1].any() for s, e in alarms))
    return [len(episodes), detected, round(detected / len(episodes), 3),
            len(alarms), len(alarms) - true_bursts, round(true_bursts / max(len(alarms), 1), 3)]

candidates = [('full (22 sensors)', err_te, err_va),
              ('cooling only (16)', sens_err[:, ci].mean(axis=1), sens_va[:, ci].mean(axis=1))]
rows = []
for name, s_te, s_va in candidates:
    for rule, thr in [('95th', np.percentile(s_va, 95)),
                      ('99th', np.percentile(s_va, 99)),
                      ('mean+3std', s_va.mean() + 3 * s_va.std())]:
        rows.append([name, rule] + event_report(thr, s_te))

print(pd.DataFrame(rows, columns=['score', 'threshold', 'episodes', 'detected', 'event_recall',
                                  'alarm_bursts', 'false_bursts', 'burst_precision']).to_string(index=False))

            score threshold  episodes  detected  event_recall  alarm_bursts  false_bursts  burst_precision
full (22 sensors)      95th        31        23         0.742           619           593            0.042
full (22 sensors)      99th        31        14         0.452           483           468            0.031
full (22 sensors) mean+3std        31        13         0.419           382           368            0.037
cooling only (16)      95th        31        20         0.645           682           661            0.031
cooling only (16)      99th        31        14         0.452           339           324            0.044
cooling only (16) mean+3std        31        13         0.419           304           290            0.046


In [9]:
import time
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support

def train_subset(cols, epochs=30, verbose=2):
    ids = [idx[c] for c in cols]
    A, B, C = Xtr[:, :, ids], Xva[:, :, ids], Xte[:, :, ids]      # keep only these sensors
    keras.utils.set_random_seed(42)
    m = keras.Sequential([
        layers.Input(shape=(24, len(ids))),
        layers.LSTM(64), layers.Dense(16),
        layers.RepeatVector(24),
        layers.LSTM(64, return_sequences=True),
        layers.TimeDistributed(layers.Dense(len(ids))),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    t0 = time.time()
    m.fit(A, A, validation_data=(B, B), epochs=epochs, batch_size=256,
          callbacks=[EarlyStopping(patience=5, restore_best_weights=True)], verbose=verbose)
    secs = time.time() - t0
    s_va = ((B - m.predict(B, batch_size=1024, verbose=0)) ** 2).mean(axis=(1, 2))
    s_te = ((C - m.predict(C, batch_size=1024, verbose=0)) ** 2).mean(axis=(1, 2))
    t99 = np.percentile(s_va, 99)                                  # threshold from validation only
    pred = s_te > t99
    p, r, f1, _ = precision_recall_fscore_support(yte, pred, average='binary', zero_division=0)
    ev = event_report(t99, s_te)
    return dict(sensors='+'.join(cols), ROC=round(roc_auc_score(yte, s_te), 3),
                PR=round(average_precision_score(yte, s_te), 4), precision=round(p, 3),
                recall=round(r, 3), events=f"{ev[1]}/{ev[0]}", false_bursts=ev[4], seconds=round(secs))

res = train_subset(['fan0_0'])
print(res)

Epoch 1/30
399/399 - 48s - 121ms/step - loss: 0.0193 - val_loss: 0.0019
Epoch 2/30
399/399 - 39s - 97ms/step - loss: 0.0063 - val_loss: 0.0015
Epoch 3/30
399/399 - 40s - 101ms/step - loss: 0.0057 - val_loss: 0.0013
Epoch 4/30
399/399 - 39s - 97ms/step - loss: 0.0042 - val_loss: 0.0012
Epoch 5/30
399/399 - 39s - 98ms/step - loss: 0.0039 - val_loss: 0.0011
Epoch 6/30
399/399 - 40s - 101ms/step - loss: 0.0033 - val_loss: 0.0011
Epoch 7/30
399/399 - 41s - 102ms/step - loss: 0.0035 - val_loss: 0.0010
Epoch 8/30
399/399 - 40s - 100ms/step - loss: 0.0031 - val_loss: 0.0011
Epoch 9/30
399/399 - 38s - 96ms/step - loss: 0.0040 - val_loss: 0.0010
Epoch 10/30


ResourceExhaustedError: Graph execution error:

Detected at node sequential_1/lstm_1_2/while/lstm_cell_1/mul defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "d:\!!DATA HP LAPTOP!!\d\5 SEM\DL PROJECT\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "d:\!!DATA HP LAPTOP!!\d\5 SEM\DL PROJECT\.venv\Lib\site-packages\traitlets\config\application.py", line 1080, in launch_instance

  File "d:\!!DATA HP LAPTOP!!\d\5 SEM\DL PROJECT\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 768, in start

  File "d:\!!DATA HP LAPTOP!!\d\5 SEM\DL PROJECT\.venv\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start

  File "C:\Users\Mital\AppData\Local\Programs\Python\Python312\Lib\asyncio\windows_events.py", line 322, in run_forever

  File "C:\Users\Mital\AppData\Local\Programs\Python\Python312\Lib\asyncio\base_events.py", line 645, in run_forever

  File "C:\

In [7]:
import json
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support

meta = json.load(open(ROOT / "models" / "final7_meta.json"))
m7 = keras.models.load_model(ROOT / "models" / "final7.keras")
idx = {c: i for i, c in enumerate(feat)}
ids = [idx[c] for c in meta["sensors"]]
print("scaler matches meta:", np.allclose(scaler.mean_[ids], meta["mean"]),
      np.allclose(scaler.scale_[ids], meta["scale"]))

Xva7, Xte7 = Xva[:, :, ids], Xte[:, :, ids]
s_va = ((Xva7 - m7.predict(Xva7, batch_size=1024, verbose=0)) ** 2).mean(axis=(1, 2))
s_te = ((Xte7 - m7.predict(Xte7, batch_size=1024, verbose=0)) ** 2).mean(axis=(1, 2))
print("threshold p99 here:", round(float(np.percentile(s_va, 99)), 6), "| saved:", round(meta["thr_p99"], 6))

rep = ~np.isin(nte, [11, 13, 14, 16, 17])            # report nodes 7, 8, 9, 10
pred = s_te[rep] > meta["thr_p99"]
p, r, _, _ = precision_recall_fscore_support(yte[rep], pred, average="binary", zero_division=0)
print("report nodes: ROC", round(roc_auc_score(yte[rep], s_te[rep]), 3),
      "PR", round(average_precision_score(yte[rep], s_te[rep]), 4),
      "precision", round(p, 3), "recall", round(r, 3))

scaler matches meta: True True
threshold p99 here: 0.010124 | saved: 0.010124
report nodes: ROC 0.724 PR 0.1004 precision 0.179 recall 0.436


In [8]:
from numpy.lib.stride_tricks import sliding_window_view

cols = meta["sensors"]
mean7 = np.array(meta["mean"], dtype="float32")
scale7 = np.array(meta["scale"], dtype="float32")

parts = []
for n, g in df[df.split == "test"].groupby("node"):
    F = g[cols].to_numpy(dtype="float32")
    L = g["lab"].to_numpy()
    tmin = ((g["timestamp"] - g["timestamp"].iloc[0]).dt.total_seconds() / 60).to_numpy()
    Fw = sliding_window_view(F, (24, len(cols)))[:, 0]
    Lw = sliding_window_view(L, 24)
    Tw = sliding_window_view(tmin, 24)
    ok = ~np.isnan(Fw).any(axis=(1, 2)) & ((Tw[:, -1] - Tw[:, 0]) == 23 * 15)
    Fw, Lw = Fw[ok], Lw[ok]
    end_idx = np.arange(23, len(g))[ok]            # row of each window's last timestamp
    R = m7.predict(Fw, batch_size=1024, verbose=0)
    err = (Fw - R) ** 2                            # (windows, 24, 7)
    part = pd.DataFrame({"node": n,
                         "timestamp": g["timestamp"].to_numpy()[end_idx],
                         "score": err.mean(axis=(1, 2)),
                         "anomaly": (Lw == 2).any(axis=1),
                         "label_known": np.isin(Lw, [0, 2]).all(axis=1)})
    for j, c in enumerate(cols):
        part["err_" + c] = err[:, :, j].mean(axis=1)
        part[c] = Fw[:, -1, j] * scale7[j] + mean7[j]     # latest reading, original units
    parts.append(part)

dash = pd.concat(parts, ignore_index=True)
dash["alarm"] = dash["score"] > meta["thr_p99"]
dash.to_parquet(ROOT / "data" / "dashboard_final7.parquet")
print(dash.shape)
print(dash.groupby("node")[["alarm", "anomaly"]].sum())

(147531, 20)
      alarm  anomaly
node                
7      1808      179
8       299      138
9       301      107
10      766     1011
11      178      151
13      133      138
14      158      151
16      188      179
17      156      138
